In [ ]:
pip install google-cloud-aiplatform

In [ ]:
pip install google-cloud-storage

In [ ]:
pip install google-cloud-bigquery

In [ ]:
import os
from google.cloud import storage

# 1. Configuration
PROJECT_ID = "jovial-coral-423517-h8"
LOCATION = "us-central1"
BUCKET_NAME = "tuc-vector-search-bucket-1"
BLOB_NAME = "books.txt"
LOCAL_FILE_PATH = r"C:\Users\hmira\GitHub-Org\teachucode-courses\Google-Gemini-ADK-Agents-Lab\Lab18-vector-search\books.txt"

storage_client = storage.Client(project=PROJECT_ID)

# 2. Create the bucket if it doesn't already exist
bucket = storage_client.bucket(BUCKET_NAME)
if not bucket.exists():
    bucket = storage_client.create_bucket(bucket, location=LOCATION)
    print(f"Bucket '{BUCKET_NAME}' created successfully in {LOCATION}.")
else:
    print(f"Bucket '{BUCKET_NAME}' already exists.")

# 3. Upload books.txt into the bucket
blob = bucket.blob(BLOB_NAME)
blob.upload_from_filename(LOCAL_FILE_PATH)
print(f"Uploaded '{LOCAL_FILE_PATH}' to gs://{BUCKET_NAME}/{BLOB_NAME}")

# 4. Download and load into memory for embeddings
text_data = blob.download_as_text()
documents = [line.strip() for line in text_data.split("\n") if line.strip()]
print(f"Loaded {len(documents)} books ready for embedding!")

In [2]:
from google.cloud import aiplatform
from vertexai.language_models import TextEmbeddingModel
from google.cloud.aiplatform_v1.types import index as index_types
import uuid

#  1. Init Vertex AI
PROJECT_ID = "jovial-coral-423517-h8"
LOCATION = "us-central1"
aiplatform.init(project=PROJECT_ID, location=LOCATION)

In [ ]:
#  2. Load text file from GCS
from google.cloud import storage

BUCKET_NAME = "tuc-vector-search-bucket-1"
BLOB_NAME = "books.txt"

storage_client = storage.Client()
bucket = storage_client.bucket(BUCKET_NAME)
blob = bucket.blob(BLOB_NAME)
text_data = blob.download_as_text()

In [4]:
# Split into small chunks (for embeddings)
documents = [line.strip() for line in text_data.split("\n") if line.strip()]
print(f"Total documents after splitting into chunks: {len(documents)}")

Total documents after splitting into chunks: 500


In [ ]:
#  3. Create embeddings
import time
from vertexai.language_models import TextEmbeddingModel
# 1. Load model
embedding_model = TextEmbeddingModel.from_pretrained("text-embedding-005")
# 2. Filter out the CSV header ("Book-Title") and any empty lines
clean_docs = [doc for doc in documents if doc and doc != "Book-Title"]
print(f"Total books to embed: {len(clean_docs)}")
# 3. Batch 100 books per API call (only ~5 calls total!)
batch_size = 100
embeddings = []
for i in range(0, len(clean_docs), batch_size):
    batch = clean_docs[i : i + batch_size]
    print(f"Embedding batch {i // batch_size + 1}: books {i + 1} to {min(i + batch_size, len(clean_docs))}...")
    
    response = embedding_model.get_embeddings(batch)
    embeddings.extend([emb.values for emb in response])
    
    # 1 second pause between batches to stay well within limits
    if i + batch_size < len(clean_docs):
        time.sleep(1.0)
# Update documents list to match the embeddings (without header)
documents = clean_docs
print(f"\nDone! Successfully created {len(embeddings)} embeddings of dimension {len(embeddings[0])}.")

In [8]:
#  4. Create a STREAM_UPDATE vector index
index_id = f"my-book-index-{uuid.uuid4().hex[:6]}"
print(f"Creating index: {index_id}")

index = aiplatform.MatchingEngineIndex.create_tree_ah_index(
    display_name=index_id,
    dimensions=len(embeddings[0]),          # 768 for text-embedding-005
    approximate_neighbors_count=10,         # Required for Tree-AH
    distance_measure_type="COSINE_DISTANCE",
    leaf_node_embedding_count=500,          # <-- Added: Configures leaf node capacity
    leaf_nodes_to_search_percent=10,        # <-- Added: Configures search percentage
    shard_size="SHARD_SIZE_SMALL",          # Smallest node size for low cost
    index_update_method="STREAM_UPDATE",    # Allows real-time vector upserts
    sync=True
)

print("Waiting for index creation to complete...")
index.wait()
print("Index created successfully!")

Creating index: my-book-index-85e3f2
Creating MatchingEngineIndex
Create MatchingEngineIndex backing LRO: projects/400638557780/locations/us-central1/indexes/6549841344196509696/operations/2415477817585172480
MatchingEngineIndex created. Resource name: projects/400638557780/locations/us-central1/indexes/6549841344196509696
To use this MatchingEngineIndex in another session:
index = aiplatform.MatchingEngineIndex('projects/400638557780/locations/us-central1/indexes/6549841344196509696')
Waiting for index creation to complete...
Index created successfully!


In [14]:
# Check if endpoint has defined
print("Endpoint defined:", endpoint is not None)
print("Deployed indexes:", endpoint.deployed_indexes)

NameError: name 'endpoint' is not defined